# Status report

By Ben Welsh

Generates basic statistics from [News Homepages database extracts](https://palewi.re/docs/news-homepages/extracts.html).

In [1]:
import json
import pandas as pd
import altair as alt
from pathlib import Path
from datetime import datetime, timedelta

In [2]:
this_dir = Path("__file__").parent.absolute()

In [3]:
sources_dir = this_dir.parent / "newshomepages" / "sources"

In [4]:
extracts_dir = this_dir.parent / "extracts" / "csv"

In [5]:
df = pd.read_csv(
    extracts_dir / "screenshot-files.csv",
    parse_dates=["mtime"],
    usecols=["identifier", "handle", "file_name", "mtime"]
)

In [8]:
df['date'] = df.mtime.dt.date

In [9]:
df["date"] = pd.to_datetime(df["date"])

How many total sites?

In [10]:
total_sites = len(df.handle.unique())

In [11]:
total_sites

200

How many total screenshots?

In [12]:
total_screenshots = len(df)

In [13]:
total_screenshots

20218

When did we start?

In [14]:
start_date = min(df.date)

In [15]:
start_date

Timestamp('2022-03-22 00:00:00')

How many screenshots in the last week?

In [16]:
today = datetime.now().date()

In [17]:
today

datetime.date(2022, 6, 29)

In [18]:
one_week_ago = today - timedelta(days=7)

In [19]:
one_week_ago

datetime.date(2022, 6, 22)

In [20]:
df_this_week = df[df.date > pd.to_datetime(one_week_ago)]

In [21]:
screenshots_this_week = len(df_this_week)

In [22]:
screenshots_this_week

3768

Write out data points

In [23]:
output = dict(
    total_sites=total_sites,
    total_screenshots=total_screenshots,
    screenshots_this_week=screenshots_this_week,
)

In [24]:
json.dump(output, open(this_dir / 'status-report.json', 'w'), indent=2)

Chart the number of sites by date

In [25]:
sites_by_date = df[['date', 'handle']].drop_duplicates().groupby("date").size().rename("sites").reset_index()

In [28]:
chart = alt.Chart(
    sites_by_date.head(len(sites_by_date)-1),
    title="Sites archived by @newshomepages",
    width=500
)

bars = chart.mark_bar(
    fill="#cecece"
).encode(
    x=alt.X("date:T", title="Date", timeUnit="yearmonthdate", axis=alt.Axis(format="%B %d", grid=False)),
    y=alt.Y("sites:Q", title="Sites"),
)

line = chart.mark_line(color='#727272', strokeWidth=3).transform_window(
    # The field to average
    rolling_mean='mean(sites)',
    # The number of values before and after the current value to include.
    frame=[-7, 0]
).encode(
    x='date:T',
    y='rolling_mean:Q'
)

(bars + line) #.save(this_dir / 'sites-by-date.svg')

alt.LayerChart(...)

Chart the number of screenshots by date

In [ ]:
screenshots_by_date = df.groupby("date").size().rename("screenshots").reset_index()

In [29]:
chart = alt.Chart(
    screenshots_by_date.head(len(screenshots_by_date)-1),
    title="Screenshots saved by @newshomepages",
    width=500
)

bars = chart.mark_bar(
    fill="#cecece"
).encode(
    x=alt.X("date:T", title="Date", timeUnit="yearmonthdate", axis=alt.Axis(format="%B %d", grid=False)),
    y=alt.Y("screenshots:Q", title="Screenshots"),
)

line = chart.mark_line(color='#727272', strokeWidth=3).transform_window(
    # The field to average
    rolling_mean='mean(screenshots)',
    # The number of values before and after the current value to include.
    frame=[-7, 0]
).encode(
    x='date:T',
    y='rolling_mean:Q'
)

(bars + line) #.save(this_dir / 'screenshots-by-date.png')

NameError: name 'screenshots_by_date' is not defined